In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from time import time
import hexaly.optimizer as hexaly
from itertools import product
import sys

In [2]:
start_time = time()

In [3]:
hexaly.HxVersion.license_content = "LICENSE_KEY = ED3A-2222-89F4B124-770D-60A55B936308D780-9506208B36204986-9B3E-E289-C66E"

In [4]:
def display_callback(optimizer, event_type):
    if event_type == hexaly.HxCallbackType.DISPLAY:
        solution = optimizer.solution
        if optimizer.model.nb_objectives > 0:
            objective_expr = optimizer.model.objectives[0]
            objective_value = solution.get_value(objective_expr)
            objective_bound = solution.get_objective_bound(0)
            optimality_gap = solution.get_objective_gap(0)
            print(f"Objective Value: {objective_value}")
            print(f"Objective Bound: {objective_bound}")
            print(f"Optimality Gap: {optimality_gap}")
        else:
            print("No objectives defined in the model.")

# Defining the speed processing of each type of station

In [5]:
#GLOBAL VARIABLES
STATIC_SPEED = 37700
PALETTE_SPEED = 57200
DYNAMIC_SPEED = 83200

# Reduce memory function on data frames

In [6]:
#function  to reduce the memory usage
def reduce_mem_usage(df, verbose=False):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)    
    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print('Mem. usage decreased to {:5.2f} Mb ({:.1f}% reduction)'.format(end_mem, 100 * (start_mem - end_mem) / start_mem))
    return df

# Read client file and clean the data

In [7]:
data_df_initial= pd.read_csv(r"C:\Users\ebelul\Downloads\CSLAP_Project_All\Heuristic_Connex_Set_Project\data\BERNER_ORDER_LINES_09-12.csv", sep=';')[['PRODUCT','ORDER','STATION']].drop_duplicates().dropna()#, nrows=20000)


#Calculate the unique number of product per station
unique_counts = data_df_initial[['PRODUCT','STATION']].groupby('PRODUCT')['STATION'].nunique().reset_index(name='numb_stat_per_prod')
#Merging the count back to the original dataframe
data_df_initial = data_df_initial.merge(unique_counts, on='PRODUCT', how='left')

#Updating the product assignment with the last station they correspond
# Step 1: Identify products with numb_stat_per_prod > 1.
products_with_multiple_stations = data_df_initial[data_df_initial['numb_stat_per_prod'] > 1]['PRODUCT'].unique()

# Step 2: For these products, find the maximum order ID and the corresponding station ID.
max_order_stations = data_df_initial[data_df_initial['PRODUCT'].isin(products_with_multiple_stations)] \
    .sort_values(by='ORDER', ascending=False) \
    .drop_duplicates(subset=['PRODUCT'], keep='first')[['PRODUCT', 'STATION']]

# Step 3: Update the station ID for all occurrences of these products.
# First, merge this information back to the original df_order.
data_df_initial = data_df_initial.merge(max_order_stations, on='PRODUCT', how='left', suffixes=('', '_max_order'))

# If STATION_max_order is not NaN (i.e., for products with multiple stations), update the STATION column.
data_df_initial['STATION'] = data_df_initial.apply(lambda x: x['STATION_max_order'] if pd.notna(x['STATION_max_order']) else x['STATION'], axis=1)

# Drop the temporary STATION_max_order column as it's no longer needed.
data_df_initial.drop(columns=['STATION_max_order', 'numb_stat_per_prod'], inplace=True)

data_df_new = data_df_initial.copy()

In [8]:
print(data_df_new)

           PRODUCT       ORDER STATION
0            10-50  8076410635   01.E4
1            10-50  8076463813   01.E4
2            10-50  8076488375   01.E4
3            10-50  8076511584   01.E4
4            10-50  8076527715   01.E4
...            ...         ...     ...
1577147  99996-200  8076701396   01.30
1577148  99996-200  8076704730   01.30
1577149  99996-200  8076710334   01.30
1577150  99996-200  8076716143   01.30
1577151  99996-200  8076739762   01.30

[1577152 rows x 3 columns]


# Remove stations as required and filterout orders that have only 5 products or less

In [9]:
stations_not_included =  ['01.Z8','01.15','01.GED']
#Filter and count occurrences at the same time
data_df_new = data_df_new[~(data_df_new['STATION'].isin(stations_not_included))]
data_df_new.loc[data_df_new['STATION'] == '01.GE4', 'STATION'] = '01.E4'
# Calculate the total number of unique orders for setting the threshold
total_unique_orders = data_df_new['ORDER'].nunique()

#Calculate the frequency for each product
data_df_new['Product_Frequency'] = data_df_new.groupby('PRODUCT')['ORDER'].transform('count')

# data_df_new.rename(columns={'STATION':'OldStationID'},inplace=True)
# uniquestations = data_df_new['OldStationID'].unique().astype(str)
stations = data_df_new['STATION'].unique().astype(str)

### Mapping if needed to camuflage naming

In [10]:
# mapping_dict = {station: f'S_{i}' for i, station in enumerate(uniquestations, start=1)}
# data_df_new['STATION'] = data_df_new['OldStationID'].map(mapping_dict)
# print(mapping_dict)
# stations_not_included = ['S_26','S_25']
# #Filter and count occurrences at the same time
# data_df_new = data_df_new[~(data_df_new['STATION'].isin(stations_not_included))
# static_stations = ['S_1','S_2','S_4']
# palette_stations = ['S_19','S_3','S_18','S_8','S_5']
# union_stat = static_stations + palette_stations
# dynamic_stations = np.array(data_df_new[~(data_df_new['STATION'].isin(union_stat))]['STATION'].drop_duplicates())

In [11]:
# Count the number of 'Location_ID' for each STATION
#Calculate the unique number of product per station
total_number_of_location_per_station = data_df_new.groupby('STATION')['PRODUCT'].nunique().to_dict()

# Count the number of initial lines for each STATION
#Calculate number of product per station
station_lines_initial = data_df_new.groupby('STATION')['PRODUCT'].count().to_dict()

In [12]:
# Calculate the number of unique 'PRODUCT' values for each 'ORDER'
order_product_counts = data_df_new.groupby('ORDER')['PRODUCT'].nunique()

# Filter 'ORDER's where the count of unique 'PRODUCT' values is more than 3
orders_to_keep = order_product_counts[order_product_counts > 5].index

# Filter the original DataFrame to keep only those rows with 'ORDER's in 'orders_to_keep'
data_df_new = data_df_new[data_df_new['ORDER'].isin(orders_to_keep)]

# Categorise stations based on their type

In [13]:
static_stations = ['01.E4','01.31','01.30']
palette_stations = ['01.01','01.02','01.03','01.04','01.05']
union_stat = static_stations + palette_stations
dynamic_stations = np.array(data_df_new[~(data_df_new['STATION'].isin(union_stat))]['STATION'].drop_duplicates())

# Specify the speed of processing on each station based on their type and the numbre of locations they have

In [14]:
speed = {s:STATIC_SPEED/total_number_of_location_per_station[s] for s in static_stations if s in stations}
speed.update({s:PALETTE_SPEED/total_number_of_location_per_station[s] for s in palette_stations if s in stations})
speed.update({s:DYNAMIC_SPEED/total_number_of_location_per_station[s] for s in dynamic_stations if s in stations})

In [15]:
print(len(data_df_new))
print(len(np.unique(data_df_new['PRODUCT'].values)))
print(len(np.unique(data_df_new['ORDER'].values)))
print(len(np.unique(data_df_new['STATION'].values)))
print(sum(list(total_number_of_location_per_station.values())))

1078598
21233
83184
24
21874


# Fromat the input data in the form of product and the list of orders it is in

In [16]:
# data_df_new = pd.read_csv("..\Data\df_order_new.csv")[['Product','ORDER','MAJ_STAT']].drop_duplicates().dropna()
# data_df_new.rename(columns= {'Product': 'PRODUCT', 'MAJ_STAT':'STATION'}, inplace=True)
# data_df_new = reduce_mem_usage(data_df_new, verbose=False)

In [17]:
df_product = data_df_new.groupby('PRODUCT').agg({
    'ORDER': list,  # Aggregate orders into a list
    'STATION': 'first'  # Assuming each product is assigned to a unique station, we take the first occurrence
}).reset_index().rename(columns={'ORDER': 'ORDER_LIST'})
df_product['Product_Frequency'] = df_product['ORDER_LIST'].apply(len)
df_product = df_product.sort_values(by='Product_Frequency', ascending=False)
df_product.set_index('PRODUCT',inplace=True)
print(df_product)

                                                   ORDER_LIST STATION  \
PRODUCT                                                                 
367176      [8076384884, 8076385206, 8076385254, 807638541...   01.07   
38278       [8076384886, 8076384959, 8076385015, 807638522...   01.06   
335506      [8076384886, 8076385004, 8076385757, 807638586...   01.13   
201071      [8076385125, 8076385150, 8076385254, 807638544...   01.07   
369964-100  [8076384780, 8076384905, 8076385245, 807638525...   01.13   
...                                                       ...     ...   
116780                                           [8076603699]   01.01   
233999                                           [8076549068]   01.E4   
116778                                           [8076411783]   01.05   
233996                                           [8076549068]   01.E4   
229027-10                                        [8076453993]   01.E4   

            Product_Frequency  
PRODUCT           

In [18]:
low_freq_prod = df_product[(df_product['Product_Frequency']<=5) & (df_product['STATION'].isin(static_stations))].index
#low_freq_prod = df_product[(df_product['Product_Frequency']<=179)].index
df_product = df_product[~df_product.index.isin(low_freq_prod)]
print(len(low_freq_prod))

5258


In [19]:
print(len(df_product))

15975


# Dataframe the presents unique orders with the list of products they contain

In [20]:
data_df_new2 = data_df_new[['PRODUCT','ORDER']].drop_duplicates()
df_order = data_df_new2[data_df_new2['PRODUCT'].isin(df_product.index)].groupby('ORDER')['PRODUCT'].apply(list).reset_index(name='PRODUCT_LIST')
df_order.set_index('ORDER',inplace=True)

## Dataframe to present unique stations with the products they contain initialy

In [21]:
data_df_new3 = data_df_new[['PRODUCT','STATION']].drop_duplicates()
df_station = data_df_new3[data_df_new3['PRODUCT'].isin(df_product.index)].groupby('STATION')['PRODUCT'].apply(list).reset_index(name='PRODUCT_ASSIGNED')
df_station.set_index('STATION',inplace=True)

# Define the locations available on each station based on the products that will go through the main process of assignment

In [22]:
total_number_of_location_per_station_2 = df_product.groupby('STATION').size().to_dict()
print(total_number_of_location_per_station_2)

{'01.01': 1090, '01.02': 1845, '01.03': 1306, '01.04': 1416, '01.05': 1644, '01.06': 111, '01.07': 95, '01.08': 87, '01.09': 284, '01.10': 273, '01.11': 224, '01.12': 8, '01.13': 42, '01.21': 642, '01.22': 95, '01.23': 115, '01.24': 109, '01.25': 130, '01.26': 141, '01.27': 110, '01.28': 495, '01.30': 1197, '01.31': 2575, '01.E4': 1941}


In [23]:
# Convert df_product and speed data into dictionaries for quick lookup
product_frequency = (
    df_product[['Product_Frequency']]
    .set_index(df_product.index)['Product_Frequency']
    .to_dict()
)
# Assuming data_df_new contains PRODUCT and STATION mapping directly
product_to_station = (
    df_product[['STATION']]
    .set_index(df_product.index)['STATION']
    .to_dict()
)

In [24]:
print(len(product_frequency))
print(len(product_to_station))

15975
15975


# Provide the initital workload of each station that should be respected during the assignment from the solver as a constraint

In [25]:
products = df_product.index
orders = list(df_product['ORDER_LIST'].explode().unique())
data_group = data_df_new[data_df_new['PRODUCT'].isin(df_product.index)]

In [26]:
total_length = {s:0 for s in stations}
count = {s:0 for s in stations}
for p in products:
    for s in stations:
        if product_to_station[p] == s:
                    total_length[s] += product_frequency[p] / speed[s]
                    count[s]+=1

print(total_length)

{'01.E4': 4716.012095490648, '01.31': 7010.807320954944, '01.02': 2935.265034965028, '01.30': 1426.016976127324, '01.05': 2406.8907692307575, '01.12': 0.4471153846153845, '01.28': 222.40281250000027, '01.04': 2166.1253146853223, '01.13': 13.480889423076919, '01.09': 201.36531250000013, '01.08': 34.57701923076922, '01.10': 201.47531249999986, '01.21': 316.0469230769226, '01.24': 38.63473557692309, '01.06': 56.735, '01.23': 43.36754807692307, '01.11': 142.05288461538458, '01.03': 1677.6665384615305, '01.01': 385.61947552447884, '01.27': 42.47115384615388, '01.26': 56.32543269230771, '01.22': 25.620312500000004, '01.25': 49.61406250000002, '01.07': 54.30950721153845}


In [27]:
print(sum(total_length[s] for s in stations))
print(sum(count[s] for s in stations))
print(sum(total_number_of_location_per_station_2[s] for s in stations))

24223.329547074656
15975
15975


In [28]:
#Scale the workload coefficent to aviod numerical instabilities in the solver process
# Define scaling factors to normalize the right-hand side to 1.
# We also avoid division by zero by using a small tolerance.
#scaling = { s: (1.0 / total_length[s] if total_length[s] > 1e-12 else 1.0) for s in stations }

# Generate the smooth start solution to provide to the solver

In [29]:
def generate_mip_start(df_station,product_to_index):
    # Approach is to initialize initial_solution_product_station and initial_solution_order_station
    initial_station_assignments_named = df_station['PRODUCT_ASSIGNED'].to_dict()

    initial_station_assignments_indices = {
    s: [product_to_index[p] for p in products]
    for s, products in initial_station_assignments_named.items()
    }
    
    return initial_station_assignments_indices

# Formulating and solving the model for the main data

In [ ]:
with hexaly.HexalyOptimizer() as optimizer:
    optimizer.param.time_limit = 72000
    model = optimizer.model
    optimizer.param.verbosity = 1
    optimizer.param.set_time_between_displays(60)
    optimizer.add_callback(hexaly.HxCallbackType.DISPLAY, display_callback)
    #optimizer.param.set_nb_threads(8)
    #change from product names to indexes to fit in the set variable 
    product_to_index = {p: idx for idx, p in enumerate(products)}

    
    # Set decision variable that shows the set of products assigned to each s
    station_products = {s: model.set(len(products)) for s in stations}

    # Constraint: Products assigned exactly once (partition)
    model.constraint(model.partition([station_products[s] for s in stations]))

    # Constraint: Maximum products per station
    for s in tqdm(stations):
        model.constraint(model.count(station_products[s]) <= total_number_of_location_per_station_2.get(s, 0))

    # Constraint: Workload limits per station
    for s in  tqdm(stations):
        workload_expr = model.sum(df_product.loc[p]['Product_Frequency']/speed[s] * model.contains(station_products[s], product_to_index[p]) for p in products)
        model.constraint(workload_expr <= total_length[s])

    # Objective: Minimize total number of station visits per order
    pbar = tqdm(total=len(orders) * len(stations), desc='Inner loop', leave=True)
    objective = 0
    for o in orders:
        prods_in_o = df_order.loc[o]['PRODUCT_LIST']
        for s in stations:   
            objective += model.sum(
                model.or_(*(model.contains(station_products[s], product_to_index[p]) for p in prods_in_o))
            )
            pbar.update(1)
    model.minimize(objective)

    model.close()

    # Generate the initial solution
    initial_station_assignments_indices = generate_mip_start(df_station,product_to_index)

    # Assign MIP start values to Hexaly variables
    for s in stations:
        for idx in initial_station_assignments_indices.get(s, []):
            station_products[s].value.add(idx) 
        
    optimizer.solve()

    station_assignment_results = {
    s: station_products[s].value for s in stations
    }
    
    index_to_product = {idx: p for p, idx in product_to_index.items()}
    
    station_assignments_named = {
        s: [index_to_product[idx] for idx in station_assignment_results[s]]
        for s in stations
    }

Inner loop: 100%|█████████▉| 1996366/1996392 [02:37<00:00, 13397.27it/s]

Preprocess model 0%

Inner loop: 100%|██████████| 1996392/1996392 [02:56<00:00, 13397.27it/s]

Push initial solution 100%
Model:  expressions = 32350037, decisions = 24, constraints = 49, objectives = 1
Param:  time limit = 72000 sec, no iteration limit, time between displays = 60

[objective direction ]:     minimize

[  0 sec,       0 itr]:       680604
[ optimality gap     ]:      100.00%
Objective Value: 680604
Objective Bound: 0
Optimality Gap: 1.0


Exception ignored on calling ctypes callback function: <function HexalyOptimizer.__init__.<locals>.<lambda> at 0x0000026120DD0430>
Traceback (most recent call last):
  File "c:\Users\ebelul\Anaconda3\envs\savoye2023\lib\site-packages\hexaly\optimizer.py", line 3688, in <lambda>
    cb = (self, None, HxCallbackType.TIME_TICKED, _hx_callback_type(lambda _1, _2, _3: None))
  File "c:\Users\ebelul\Anaconda3\envs\savoye2023\lib\site-packages\hexaly\optimizer.py", line 381, in _hx_signal_handler
    _chained_signal_handler[sig](sig, frame)
KeyboardInterrupt: 


KeyboardInterrupt: 

: 

# Format the assignment of products in the form of a data frame and save it

In [ ]:
df_assignments = pd.DataFrame([
    {'PRODUCT_ID': product, 'STATION': station}
    for station, products in station_assignments_named.items()
    for product in products
])

In [ ]:
df_assignments.to_csv('product_assignment_Hexaly_set_v1.csv')

In [ ]:
end_time = time() - start_time
print(end_time)

In [ ]:
stations_not_included = ['01.Z8']
#Filter and count occurrences at the same time
data_df_initial = data_df_initial[~(data_df_initial['STATION'].isin(stations_not_included))]
data_df_initial.loc[data_df_initial['STATION'] == '01.GE4', 'STATION'] = '01.E4'
# data_df_initial.rename(columns={'STATION':'OldStationID'},inplace=True)
# data_df_initial['STATION'] = data_df_initial['OldStationID'].map(mapping_dict)

data_df_initial['Product_Frequency'] = data_df_initial.groupby('PRODUCT')['ORDER'].transform('count')

In [ ]:
print(len(np.unique(data_df_initial['ORDER'])))
print(len(np.unique(data_df_initial['PRODUCT'])))
print(len(np.unique(data_df_initial['STATION'])))

In [ ]:
print(df_assignments)

# Analyse the assignment and comparisons with initial assignment

In [ ]:
product_station_df = df_assignments.copy()
product_station_df.rename(columns={"STATION":"StationID_P","PRODUCT_ID":"PRODUCT"},inplace=True)
product_station_df.head(5)

In [ ]:
df_final = pd.merge(data_df_initial,product_station_df, on='PRODUCT',how='left')[['PRODUCT','STATION','StationID_P','ORDER','Product_Frequency']]
df_final['StationID_P'].fillna(df_final['STATION'], inplace=True)
print(df_final)

# Total and Mean number of order station passe

## Initial assignment

In [ ]:
df_order_s1 = df_final[['STATION','ORDER']].drop_duplicates()
df_order_s1['cnt'] = df_order_s1.groupby('ORDER')['STATION'].transform('nunique')
df_order_s1_cnt = df_order_s1[['cnt','ORDER']].drop_duplicates()
s1 = df_order_s1_cnt['cnt'].sum()
m1 = df_order_s1_cnt['cnt'].mean()
print(s1,m1)

## Solver assignment

In [ ]:
df_order_s2 = df_final[['StationID_P','ORDER']].drop_duplicates()
df_order_s2['cnt'] = df_order_s2.groupby('ORDER')['StationID_P'].transform('nunique')
df_order_s2_cnt = df_order_s2[['cnt','ORDER']].drop_duplicates()
s2 = df_order_s2_cnt['cnt'].sum()
m2 = df_order_s2_cnt['cnt'].mean()
print(s2,m2)

In [ ]:
df_order_s1_cnt = df_order_s1_cnt['cnt'].value_counts(dropna=False)
df_order_s2_cnt = df_order_s2_cnt['cnt'].value_counts(dropna=False)

# Function to visualise the number of orders that pass through a respective total number of stations

In [ ]:
import matplotlib.pyplot as plt
# Align the indices
all_indices = sorted(set(df_order_s1_cnt.index) | set(df_order_s2_cnt.index))  # Union of indices from both datasets
df_order_s1_cnt = df_order_s1_cnt.reindex(all_indices).fillna(0)  # Reindex and fill missing values with 0
df_order_s2_cnt = df_order_s2_cnt.reindex(all_indices).fillna(0)  # Reindex and fill missing values with 0

print(all_indices)
# Set up the figure and axis
plt.figure(figsize=(10, 5))
ax = plt.gca()


# Width of the bars
width = 0.35


# Positions for the bars
indices = np.arange(len(all_indices))


# Plotting
rects1 = ax.bar(indices - width/2, df_order_s1_cnt, width, label='Original', alpha=0.6)
rects2 = ax.bar(indices + width/2, df_order_s2_cnt, width, label='Our Solution', alpha=0.6)


# Add labels and title
plt.xlabel('Number of Stations')
plt.ylabel('Number of Orders')
plt.title('Number of Orders per Stations Numbers')


# Add x-ticks and labels
plt.xticks(indices, all_indices)
# Add a legend
plt.legend()
plt.savefig('Hexaly'+'_'+'number_of_orders_per_station_numbers.png')


plt.show()

# Function to visualise the workload on each station

In [ ]:
def plot_final(tmp_final,xlabel,ylabel,title, sort_by='STATION', ascending=True):


    # Set up the figure and axis
    plt.figure(figsize=(10, 5))
    ax = plt.gca()


    # Width of the bars
    width = 0.35
    
    # Sorting tmp_final based on the specified sort_by column
    tmp_final_sorted = tmp_final.sort_values(by=sort_by, ascending=ascending)


    # Positions for the bars
    stations = tmp_final['STATION'].astype(str).values
    indices = np.arange(len(stations))


    # Plotting
    rects1 = ax.bar(indices - width/2, tmp_final['cnt_lines'], width, label='Original', alpha=0.6)
    rects2 = ax.bar(indices + width/2, tmp_final['cnt_lines_New'], width, label='Our Solution', alpha=0.6)


    # Add labels and title
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)


    # Add x-ticks and labels
    plt.xticks(indices, stations, rotation=45)


    # Add a legend
    plt.legend()


    plt.savefig('Hexaly'+'_'+'number_of_lines_per_station.png')
    plt.show()

In [ ]:
df_order_lines_plot = df_final.copy()
df_order_lines_plot['cnt_lines_New'] = df_order_lines_plot[['PRODUCT','StationID_P']].groupby('StationID_P').transform('count')
df_order_lines_plot['cnt_lines'] = df_order_lines_plot[['PRODUCT','STATION']].groupby('STATION').transform('count')
tmp_lines = df_order_lines_plot[['StationID_P','cnt_lines_New']].drop_duplicates()
tmp_lines.rename(columns={'StationID_P':'STATION'}, inplace=True)
tmp_lines = tmp_lines.merge(df_order_lines_plot[['STATION','cnt_lines']].drop_duplicates(), on='STATION', how='outer').fillna(0)
tmp_lines.sort_values(by='cnt_lines', ascending=True, inplace=True)
xlabel = 'Station ID'
ylabel = 'Number of lines'
title = 'Number of Lines per Station'
plot_final(tmp_lines, xlabel, ylabel, title, sort_by='cnt_lines', ascending=False)

# Visualising the number of products assigned on each station

In [ ]:
df_order_ref_plot = df_final.copy()
df_order_ref_plot['cnt_lines_New'] = df_order_ref_plot[['PRODUCT','StationID_P']].groupby('StationID_P').transform('nunique')
df_order_ref_plot['cnt_lines'] = df_order_ref_plot[['PRODUCT','STATION']].groupby('STATION').transform('nunique')
tmp_refs = df_order_ref_plot[['StationID_P','cnt_lines_New']].drop_duplicates()
tmp_refs.rename(columns={'StationID_P':'STATION'}, inplace=True)
tmp_refs = tmp_refs.merge(df_order_ref_plot[['STATION','cnt_lines']].drop_duplicates(), on='STATION', how='outer').fillna(0)
tmp_refs.sort_values(by='cnt_lines', ascending=True, inplace=True)
xlabel = 'Station ID'
ylabel = 'Number of references'
title = 'Number of Products per Station'
plot_final(tmp_refs, xlabel, ylabel, title, sort_by='cnt_lines', ascending=False)

In [ ]:
print(tmp_refs)

In [ ]:
product_frequency = df_final[['PRODUCT','Product_Frequency']].drop_duplicates().set_index('PRODUCT')['Product_Frequency'].to_dict()
station_product_initial = df_final.groupby('STATION')['PRODUCT'].apply(set).to_dict()
station_product_solution = df_final.groupby('StationID_P')['PRODUCT'].apply(set).to_dict()

In [ ]:
total_length_initial={s:0 for s in stations}
total_length_solution={s:0 for s in stations}

for s in stations:
    for p1 in station_product_initial[s]:
        total_length_initial[s] += product_frequency[p1]/speed[s]
        
    for p2 in station_product_solution[s]:
        total_length_solution[s] += product_frequency[p2]/speed[s]

    print(f"Length constraint value = {total_length_solution[s]}, Limit = {total_length_initial[s]}, on station {s}")

In [ ]:
print(np.sum(total_length_solution[s] for s in stations))

In [ ]:
print(np.sum(total_length_initial[s] for s in stations))